# Chapter 4 — Implementing a GPT model from scratch to generate text

This notebook assembles the components developed in earlier chapters into a GPT-style language model. The implementation begins with
a validated architecture configuration and a shape-preserving dummy model before replacing each placeholder with a real Transformer
component.

## Learning goals

- represent model hyperparameters with a typed, validated configuration;
- trace token IDs through embeddings, Transformer blocks, normalization, and logits;
- preserve `(batch, tokens, embedding)` shapes through the Transformer blocks; and
- produce one vocabulary-logit vector per input token.

## 4.1 Defining a typed and validated GPT configuration

A plain dictionary accepts misspelled keys, missing values, invalid ranges, and incompatible dimensions until much later in model
construction. `GPTConfig` replaces string-based access such as `cfg["emb_dim"]` with typed attribute access such as `cfg.emb_dim`.

Pydantic validates values when the object is created. Positive dimensions, a dropout rate in `[0, 1)`, and divisibility of `emb_dim`
by `n_heads` become explicit architecture contracts. Freezing the model prevents accidental mutation after layers are initialized.

In [ ]:
from typing import Self

from pydantic import BaseModel, ConfigDict, Field, model_validator


class GPTConfig(BaseModel):
    """Validated architecture settings for a GPT-style language model."""

    # Configuration objects behave like immutable architecture specifications.
    model_config = ConfigDict(frozen=True)

    vocab_size: int = Field(gt=0, description="Number of tokenizer vocabulary entries")
    context_length: int = Field(gt=0, description="Maximum supported token sequence length")
    emb_dim: int = Field(gt=0, description="Token embedding and model dimension, also called d_model")
    n_heads: int = Field(gt=0, description="Number of parallel attention heads")
    n_layers: int = Field(gt=0, description="Number of Transformer blocks")
    drop_rate: float = Field(ge=0.0, lt=1.0, description="Dropout probability")
    qkv_bias: bool = Field(description="Whether QKV projections include bias terms")

    @model_validator(mode="after")
    def validate_attention_dimensions(self) -> Self:
        """Require every attention head to receive an equal feature width."""
        if self.emb_dim % self.n_heads != 0:
            raise ValueError("emb_dim must be divisible by n_heads")
        return self

    @property
    def head_dim(self) -> int:
        """Return the query, key, and value width assigned to one head."""
        return self.emb_dim // self.n_heads


# GPT-2 small / 124M uses a model dimension of 768 and 12 attention heads.
GPT_CONFIG_124M = GPTConfig(
    vocab_size=50257,
    context_length=1024,
    emb_dim=768,
    n_heads=12,
    n_layers=12,
    drop_rate=0.1,
    qkv_bias=False,
)

### GPT-2 small configuration

`GPT_CONFIG_124M` describes the smallest GPT-2 architecture: a 50,257-token vocabulary, context length 1,024, embedding dimension 768,
12 heads, and 12 Transformer blocks. Its derived `head_dim` is `768 / 12 = 64`.

Use `GPT_CONFIG_124M.model_dump()` only when an external API specifically requires a dictionary; model code should prefer typed
attributes.

## 4.2 Creating shape-preserving placeholders

Before implementing full Transformer blocks and layer normalization, identity modules let us assemble and test the outer GPT data
flow. They expose the same typed interfaces as the eventual components while returning inputs unchanged.

In [ ]:
import torch
from torch import nn


class DummyTransformerBlock(nn.Module):
    """Placeholder Transformer block that preserves its input unchanged."""

    def __init__(self, cfg: GPTConfig) -> None:
        """Accept the final block interface without creating operations yet."""
        super().__init__()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Return a tensor with the same values and shape as the input."""
        return x


class DummyLayerNorm(nn.Module):
    """Placeholder final normalization that acts as an identity function."""

    def __init__(self, normalized_shape: int, eps: float = 1e-5) -> None:
        """Accept the final normalization interface without parameters yet."""
        super().__init__()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Return a tensor with the same values and shape as the input."""
        return x

## 4.3 Assembling the dummy GPT model

The model maps token IDs through these stages:

```text
token IDs                     (B, T)
token embeddings              (B, T, emb_dim)
positional embeddings         (T, emb_dim)
combined token representations (B, T, emb_dim)
Transformer block stack       (B, T, emb_dim)
final normalization           (B, T, emb_dim)
vocabulary logits             (B, T, vocab_size)
```

Positional embeddings broadcast across the batch. The output head changes only the final feature dimension, producing one vector of
next-token scores for every input position.

In [ ]:
class DummyGPTModel(nn.Module):
    """Trace GPT tensor shapes while Transformer internals remain placeholders."""

    def __init__(self, cfg: GPTConfig) -> None:
        """Create embeddings, placeholder blocks, normalization, and output head."""
        super().__init__()
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.emb_dim)
        self.pos_emb = nn.Embedding(cfg.context_length, cfg.emb_dim)
        self.drop_emb = nn.Dropout(cfg.drop_rate)
        self.trf_blocks = nn.Sequential(
            *[DummyTransformerBlock(cfg) for _ in range(cfg.n_layers)]
        )
        self.final_norm = DummyLayerNorm(cfg.emb_dim)
        self.out_head = nn.Linear(cfg.emb_dim, cfg.vocab_size, bias=False)

    def forward(self, in_idx: torch.Tensor) -> torch.Tensor:
        """Map token IDs `(B, T)` to vocabulary logits `(B, T, vocab_size)`."""
        # in_idx: (batch_size, seq_len)
        _, seq_len = in_idx.shape

        # tok_embeds: (batch_size, seq_len, emb_dim)
        tok_embeds = self.tok_emb(in_idx)
        # pos_embeds: (seq_len, emb_dim), broadcast across the batch dimension
        pos_embeds = self.pos_emb(
            torch.arange(seq_len, device=in_idx.device)
        )

        # x: (batch_size, seq_len, emb_dim)
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        # Placeholder blocks and normalization preserve x's shape.
        x = self.trf_blocks(x)
        x = self.final_norm(x)

        # logits: (batch_size, seq_len, vocab_size)
        logits = self.out_head(x)
        return logits

## Checkpoint

The architecture is now represented by an immutable, validated `GPTConfig`, and the dummy model establishes the complete tensor-shape
contract from token IDs to vocabulary logits. Subsequent sections can replace placeholders without changing that outer interface.